# Genome-wide QC + thinning (Google Batch / dsub)

Batch version of `03a` -- one dedicated VM per chromosome, avoids the gcsfuse-mount contention `03a`'s concurrent driver hits.

ACAF access notes:
- Default Batch image has no `gcloud`/`gsutil`.
- dsub's own `--input` can't pass a billing project -- fails on this Requester Pays bucket.
- Fix: `--image gcr.io/google.com/cloudsdktool/cloud-sdk:slim` + `gsutil -u "$PROJECT_ID" cp ...` inside the script.

## Prerequisites

`dsub` installed, `gcloud` authenticated. Submits from this VM; work runs on separate Batch VMs.

In [ ]:
%%bash
set -e

if ! command -v dsub >/dev/null 2>&1; then
  pip install --quiet dsub
fi
dsub --version

echo "--- gcloud config ---"
gcloud config list --format='text(core.project,compute.region)' 2>&1 || true

## Inputs

Same `PROJECT_ID`/`REGION`/`SERVICE_ACCOUNT`/`NETWORK`/`SUBNETWORK`/`WORKSPACE_BUCKET_GS` as `03b_grm_shard_batch_submit.ipynb`. `ACAF_BUCKET_GS` confirmed via `ps -eo pid,args | grep gcsfuse`.

In [ ]:
import os

# same values 03b_grm_shard_batch_submit.ipynb already confirmed working -- reused,
# not re-derived
PROJECT_ID = "wb-swift-sprout-7231"
REGION = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
WORKSPACE_BUCKET_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning (selecting the ACAF data source
# below, and filename/log tags) throughout this notebook. Fixed literal, not
# derived from CDR_VERSION, so this project's outputs are self-describing
# rather than sitting under a folder named after whatever CDR version happens
# to be active.
PROJECT_DIR = "covariance_v9"

# Confirmed via `ps -eo pid,args | grep gcsfuse` on the real VM: the mount for
# ~/workspace/cdrv9/vwb-aou-datasets-controlled-v9 is
# `gcsfuse --billing-project wb-swift-sprout-7231 ... vwb-aou-datasets-controlled ...`
# -- bucket name has no version suffix; v9's own subpath comes after it, matching
# the local mount's own subpath structure. The `--billing-project` flag on that
# gcsfuse invocation confirms this is a Requester Pays bucket. The default Batch
# worker image has no `gcloud` binary at all (confirmed from a real failed run:
# `gcloud: command not found`), so there is no in-script way to pass a billing
# project explicitly -- the only I/O mechanism available is dsub's own `--input`
# staging (Stage 0 below tests whether that mechanism itself can read this
# Requester Pays bucket, since that's genuinely unconfirmed).
ACAF_BUCKET_GS = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/pgen"

# Which base ancestry group to QC/thin -- must match 01_premade_label_filter.ipynb's
# own BASE_GROUP values ("eur", "afr").
BASE_GROUP = "eur"   # <-- change this and rerun for "eur" and "afr"

ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
ANCESTRY_BUCKET_DIR_GS = f"{WORKSPACE_BUCKET_GS}/{PROJECT_DIR}/01_ancestry_filtering"

PREMADE_KEEP_PATH = f"{ANCESTRY_BUCKET_DIR}/premade_label_{CDR_VERSION}/premade_keep_ids_{BASE_GROUP}.txt"
PREMADE_KEEP_PATH_GS = f"{ANCESTRY_BUCKET_DIR_GS}/premade_label_{CDR_VERSION}/premade_keep_ids_{BASE_GROUP}.txt"
assert os.path.isfile(PREMADE_KEEP_PATH), f"premade-label keep-list not found: {PREMADE_KEEP_PATH!r} -- run 01_premade_label_filter.ipynb first"

# same output location 03a_genome_wide_qc_thinning_merge.ipynb writes to, so its
# own "Merge chromosomes" section works unchanged on whichever chromosomes were
# produced here instead
BUCKET_DIR_GS = f"{ANCESTRY_BUCKET_DIR_GS}/genome_wide_panel_{BASE_GROUP}"
PLINK_BIN_GS = f"{BUCKET_DIR_GS}/bin/plink2"

THIN_P = 0.2   # fixed from prior chr22 calibration, same as 03a's

# per-task machine. Real Stage 2 run: 18/22 chromosomes succeeded on
# n1-standard-8, but the 4 largest (chr1-4) all died silently mid-download
# with no script-level error -- n1-standard-8 has only 30GB of *physical*
# RAM, while MEMORY_MB below was passed to plink2's --memory as 32768 (32GB),
# already over the machine's real ceiling before plink2 even ran; combined
# with --disk-size 100 being sized against chr22-scale files, not chr1's
# much larger unthinned .pgen. Bumped both machine size and disk headroom so
# MEMORY_MB has real room under the VM's actual RAM instead of exceeding it.
MACHINE_VCPUS = 16
MEMORY_MB = 55000

print(ACAF_BUCKET_GS)
print(BUCKET_DIR_GS)

## Stage the plink2 binary (one-time)

No `wget`/`curl` on the default image -- stage once, localize via `--input`.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

"$BIN_DIR/plink2" --version

In [ ]:
%%bash -s "$PLINK_BIN_GS"
set -e
PLINK_BIN_GS=$1

local_plink="$HOME/bin/plink2"
if [ ! -x "$local_plink" ]; then
  echo "no local plink2 at $local_plink -- run the cell above first" >&2
  exit 1
fi

gcloud storage cp "$local_plink" "$PLINK_BIN_GS"
gcloud storage ls -l "$PLINK_BIN_GS"

## Stage 0 -- confirm `gsutil -u` can read the ACAF bucket

Minimal task, no plink. If it fails, check the real log for VPC-SC/IAM (no fix) vs. something else (fixable).

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$ACAF_BUCKET_GS" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
ACAF_BUCKET_GS=$8
ANCESTRY_BUCKET_DIR_GS=$9

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "acaf-access-check" \
  --env PROJECT_ID="$PROJECT_ID" \
  --env ACAF_TEST_PATH="${ACAF_BUCKET_GS}/acaf_threshold.chr22.pvar" \
  --command '
    echo "Checking read access to: $ACAF_TEST_PATH (billing project: $PROJECT_ID)"
    if gsutil -u "$PROJECT_ID" stat "$ACAF_TEST_PATH"; then
      echo "SUCCESS: gsutil -u can read the ACAF bucket from this image"
    else
      echo "FAILED: gsutil -u cannot read the ACAF bucket -- check status-detail/log: VPC-SC/IAM (no fix) vs. something else" >&2
      exit 1
    fi
  ' \
  > /tmp/acaf_check_job_id.txt

cat /tmp/acaf_check_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(tail -1 /tmp/acaf_check_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Stage 1 -- single-chromosome validation

Run after Stage 0 succeeds. chr22, `--image`+`gsutil -u` for ACAF, `--input` for KEEP_PATH/PLINK_BIN. Writes `chr{N}_thinned_{CDR_VERSION}_{BASE_GROUP}.{pgen,pvar,psam}`.

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$ACAF_BUCKET_GS" "$PREMADE_KEEP_PATH_GS" "$PLINK_BIN_GS" "$BUCKET_DIR_GS" "$THIN_P" "$MACHINE_VCPUS" "$MEMORY_MB" "$BASE_GROUP" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
ACAF_BUCKET_GS=$8
PREMADE_KEEP_PATH_GS=$9
PLINK_BIN_GS=${10}
BUCKET_DIR_GS=${11}
THIN_P=${12}
MACHINE_VCPUS=${13}
MEMORY_MB=${14}
BASE_GROUP=${15}
ANCESTRY_BUCKET_DIR_GS=${16}

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"
CHR=22
OUT_NAME="chr${CHR}_thinned_${CDR_VERSION}_${BASE_GROUP}"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "qc-thin-chr22-validation" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input KEEP_PATH="$PREMADE_KEEP_PATH_GS" \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --env CHR_PGEN_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.pgen" \
  --env CHR_PVAR_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.pvar" \
  --env CHR_PSAM_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.psam" \
  --env THIN_P="$THIN_P" \
  --env OUT_NAME="$OUT_NAME" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    # Requester Pays bucket -- gsutil -u required, same command Stage 0
    # already proved works from this image
    gsutil -u "$PROJECT_ID" cp "$CHR_PGEN_GS" chr.pgen
    gsutil -u "$PROJECT_ID" cp "$CHR_PVAR_GS" chr.pvar
    gsutil -u "$PROJECT_ID" cp "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --keep "$KEEP_PATH" \
      --thin '"$THIN_P"' \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --maf 0.01 \
      --hwe 1e-6 0.001 keep-fewhet \
      --geno 0.05 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  > /tmp/qc_validation_job_id.txt

cat /tmp/qc_validation_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3
JOB_ID=$(tail -1 /tmp/qc_validation_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

echo
echo "--- output files ---"
gcloud storage ls -l "${BUCKET_DIR_GS}/" 2>/dev/null | grep chr22 || echo "(none yet -- check task status above)"

## Stage 2 -- full run (all 22 chromosomes)

Run after Stage 1 succeeds. One task per chromosome, `--tasks` TSV.

First real run: chr1-4 (largest) failed on `n1-standard-8`/`MEMORY_MB=32768` (over the machine's real 30GB RAM) with `--disk-size 100`. Fixed: `n1-standard-16`/`MEMORY_MB=55000`/`--disk-size 300`.

In [ ]:
CHR_LENGTHS = {
    1: 248_956_422, 2: 242_193_529, 3: 198_295_559, 4: 190_214_555,
    5: 181_538_259, 6: 170_805_979, 7: 159_345_973, 8: 145_138_636,
    9: 138_394_717, 10: 133_797_422, 11: 135_086_622, 12: 133_275_309,
    13: 114_364_328, 14: 107_043_718, 15: 101_991_189, 16: 90_338_345,
    17: 83_257_441, 18: 80_373_285, 19: 58_617_616, 20: 64_444_167,
    21: 46_709_983, 22: 50_818_468,
}
CHRS_LARGEST_FIRST = sorted(CHR_LENGTHS, key=CHR_LENGTHS.get, reverse=True)

# gs:// paths passed as plain --env strings, not --input -- Stage 0/1 confirmed
# gsutil -u "$PROJECT_ID" cp inside the script (on the cloud-sdk:slim image) is
# what actually works against this Requester Pays bucket, so Stage 2 uses the
# same mechanism
TASKS_PATH = "/tmp/qc_thin_tasks.tsv"
with open(TASKS_PATH, "w") as f:
    f.write("--env CHR\t--env CHR_PGEN_GS\t--env CHR_PVAR_GS\t--env CHR_PSAM_GS\t--env OUT_NAME\n")
    for chr_num in CHRS_LARGEST_FIRST:
        out_name = f"chr{chr_num}_thinned_{CDR_VERSION}_{BASE_GROUP}"
        f.write(
            f"{chr_num}\t{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pgen\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pvar\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.psam\t{out_name}\n"
        )

print(open(TASKS_PATH).read())

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$PREMADE_KEEP_PATH_GS" "$PLINK_BIN_GS" "$BUCKET_DIR_GS" "$THIN_P" "$MACHINE_VCPUS" "$MEMORY_MB" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
PREMADE_KEEP_PATH_GS=$8
PLINK_BIN_GS=$9
BUCKET_DIR_GS=${10}
THIN_P=${11}
MACHINE_VCPUS=${12}
MEMORY_MB=${13}
ANCESTRY_BUCKET_DIR_GS=${14}

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "qc-thin-genome-wide" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input KEEP_PATH="$PREMADE_KEEP_PATH_GS" \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    # Requester Pays bucket -- gsutil -u required, same mechanism
    # Stage 0/1 already proved works on this image
    gsutil -u "$PROJECT_ID" cp "$CHR_PGEN_GS" chr.pgen
    gsutil -u "$PROJECT_ID" cp "$CHR_PVAR_GS" chr.pvar
    gsutil -u "$PROJECT_ID" cp "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --keep "$KEEP_PATH" \
      --thin '"$THIN_P"' \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --maf 0.01 \
      --hwe 1e-6 0.001 keep-fewhet \
      --geno 0.05 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  --tasks /tmp/qc_thin_tasks.tsv \
  > /tmp/qc_genome_wide_job_id.txt

cat /tmp/qc_genome_wide_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(cat /tmp/qc_genome_wide_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Resubmit missing chromosomes only

Checks the bucket directly (not job history) for what's missing, resubmits just those. Safe to rerun anytime.

In [ ]:
# gcsfuse-mounted local path to the same directory Batch's --output-recursive
# writes to -- checking file existence here (not job status) is what actually
# tells you whether a chromosome is done, since a chromosome could have
# succeeded on an earlier job than the one you're looking at in dstat
BUCKET_DIR = os.path.join(ANCESTRY_BUCKET_DIR, f"genome_wide_panel_{BASE_GROUP}")

def chr_is_done(chr_num):
    prefix = os.path.join(BUCKET_DIR, f"chr{chr_num}_thinned_{CDR_VERSION}_{BASE_GROUP}")
    return all(os.path.isfile(f"{prefix}.{ext}") for ext in ("pgen", "pvar", "psam"))

MISSING_CHRS = [c for c in range(1, 23) if not chr_is_done(c)]
print(f"{22 - len(MISSING_CHRS)}/22 chromosomes already persisted; missing: {MISSING_CHRS}")

RESUBMIT_TASKS_PATH = "/tmp/qc_thin_tasks_resubmit.tsv"
with open(RESUBMIT_TASKS_PATH, "w") as f:
    f.write("--env CHR\t--env CHR_PGEN_GS\t--env CHR_PVAR_GS\t--env CHR_PSAM_GS\t--env OUT_NAME\n")
    for chr_num in MISSING_CHRS:
        out_name = f"chr{chr_num}_thinned_{CDR_VERSION}_{BASE_GROUP}"
        f.write(
            f"{chr_num}\t{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pgen\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pvar\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.psam\t{out_name}\n"
        )

if MISSING_CHRS:
    print(open(RESUBMIT_TASKS_PATH).read())
else:
    print("Nothing to resubmit -- all 22 chromosomes already persisted. Skip the next cell and go to the merge section.")

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$PREMADE_KEEP_PATH_GS" "$PLINK_BIN_GS" "$BUCKET_DIR_GS" "$THIN_P" "$MACHINE_VCPUS" "$MEMORY_MB" "$ANCESTRY_BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
PREMADE_KEEP_PATH_GS=$8
PLINK_BIN_GS=$9
BUCKET_DIR_GS=${10}
THIN_P=${11}
MACHINE_VCPUS=${12}
MEMORY_MB=${13}
ANCESTRY_BUCKET_DIR_GS=${14}

# skip submitting an empty --tasks job if the previous cell found nothing missing
if [ "$(tail -n +2 /tmp/qc_thin_tasks_resubmit.tsv | wc -l)" -eq 0 ]; then
  echo "Nothing to resubmit -- all chromosomes already persisted."
  exit 0
fi

LOGGING_GS="${ANCESTRY_BUCKET_DIR_GS}/dsub_logs"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:slim" \
  --name "qc-thin-resubmit" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 300 \
  --input KEEP_PATH="$PREMADE_KEEP_PATH_GS" \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    gsutil -u "$PROJECT_ID" cp "$CHR_PGEN_GS" chr.pgen
    gsutil -u "$PROJECT_ID" cp "$CHR_PVAR_GS" chr.pvar
    gsutil -u "$PROJECT_ID" cp "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --keep "$KEEP_PATH" \
      --thin '"$THIN_P"' \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --maf 0.01 \
      --hwe 1e-6 0.001 keep-fewhet \
      --geno 0.05 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  --tasks /tmp/qc_thin_tasks_resubmit.tsv \
  > /tmp/qc_resubmit_job_id.txt

cat /tmp/qc_resubmit_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(cat /tmp/qc_resubmit_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Next steps

Once all 22 are `SUCCESS`, run the merge section below.

## Merge chromosomes into a genome-wide pfile

Concatenates the 22 QC'd trios via `--pmerge-list`, plus a `--make-bed` export (PLINK 1.9's `--parallel` split, used by the GRM step, needs bed/bim/fam).

In [ ]:
import shutil

# gcsfuse-mounted local path to the same directory Batch's --output-recursive
# wrote to (BUCKET_DIR_GS's local equivalent) -- merging reads/writes local
# disk, not the bucket directly, same convention as 03a
BUCKET_DIR = os.path.join(ANCESTRY_BUCKET_DIR, f"genome_wide_panel_{BASE_GROUP}")
LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

# Real failure: --pmerge-list errored with "Invalid variant count in .pgen
# file" -- root cause was a truncated local .pgen left over from an earlier
# interrupted run of this exact cell (kernel restart mid-copy). The original
# `if not os.path.isfile(local_path)` skip only checked existence, not
# completeness, so the truncated file was silently treated as already copied
# on every rerun. Comparing sizes against the bucket source (still cheap --
# os.path.getsize doesn't read file contents) catches that case and
# re-copies instead of trusting a partial file.
for chr_num in range(1, 23):
    for ext in ("pgen", "pvar", "psam"):
        bucket_path = os.path.join(BUCKET_DIR, f"chr{chr_num}_thinned_{CDR_VERSION}_{BASE_GROUP}.{ext}")
        local_path = os.path.join(LOCAL_WORK_DIR, f"chr{chr_num}_thinned_{CDR_VERSION}_{BASE_GROUP}.{ext}")
        assert os.path.isfile(bucket_path), f"missing persisted chromosome output: {bucket_path!r} -- rerun Stage 2/resubmit for chr{chr_num} first"
        needs_copy = not os.path.isfile(local_path) or os.path.getsize(local_path) != os.path.getsize(bucket_path)
        if needs_copy:
            shutil.copy(bucket_path, local_path)

print("All 22 chromosome trios present in local scratch (size-verified against the bucket).")

In [ ]:
MERGE_LIST_PATH = os.path.join(LOCAL_WORK_DIR, f"chr_merge_list_{CDR_VERSION}_{BASE_GROUP}.txt")
with open(MERGE_LIST_PATH, "w") as f:
    for chr_num in range(1, 23):
        f.write(os.path.join(LOCAL_WORK_DIR, f"chr{chr_num}_thinned_{CDR_VERSION}_{BASE_GROUP}") + "\n")

MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, f"genome_wide_thinned_{CDR_VERSION}_{BASE_GROUP}")
BED_PREFIX = f"{MERGED_PREFIX}_bed"
PLINK2_LOCAL_BIN = os.path.expanduser("~/bin/plink2")
assert os.path.isfile(PLINK2_LOCAL_BIN), f"plink2 not found at {PLINK2_LOCAL_BIN!r} -- run the 'Stage the plink2 binary' cell above first"

print(MERGE_LIST_PATH)
print(MERGED_PREFIX)

In [ ]:
%%bash -s "$MERGE_LIST_PATH" "$MERGED_PREFIX" "$BED_PREFIX" "$PLINK2_LOCAL_BIN"
set -e
MERGE_LIST_PATH=$1
MERGED_PREFIX=$2
BED_PREFIX=$3
PLINK2_LOCAL_BIN=$4

time "$PLINK2_LOCAL_BIN" \
  --pmerge-list "$MERGE_LIST_PATH" \
  --make-pgen \
  --out "$MERGED_PREFIX"

echo "Genome-wide variant count:"
grep -vc '^##' "${MERGED_PREFIX}.pvar"
echo "Sample count:"
wc -l < "${MERGED_PREFIX}.psam"

# PLINK 1.9 (the GRM step) doesn't read pgen -- export a bed/bim/fam copy too
time "$PLINK2_LOCAL_BIN" \
  --pfile "$MERGED_PREFIX" \
  --make-bed \
  --out "$BED_PREFIX"

ls -lh "${MERGED_PREFIX}".* "${BED_PREFIX}".*

In [ ]:
%%bash -s "$MERGED_PREFIX" "$BED_PREFIX" "$BUCKET_DIR"
set -e
MERGED_PREFIX=$1
BED_PREFIX=$2
BUCKET_DIR=$3

mkdir -p "$BUCKET_DIR"
cp "${MERGED_PREFIX}".pgen "${MERGED_PREFIX}".pvar "${MERGED_PREFIX}".psam "${MERGED_PREFIX}".log "$BUCKET_DIR/"
cp "${BED_PREFIX}".bed "${BED_PREFIX}".bim "${BED_PREFIX}".fam "${BED_PREFIX}".log "$BUCKET_DIR/"

ls -lh "$BUCKET_DIR"/"$(basename "$MERGED_PREFIX")".* "$BUCKET_DIR"/"$(basename "$BED_PREFIX")".*